# Logistics Data Analytics Intern — Weeks 1–3
## End-to-End Logistics Strategy, Data Preparation, Advanced Analysis & Visualization

**Dataset:** DataCo Supply Chain Dataset  
**Tools:** Python, Pandas, NumPy, Matplotlib, PostgreSQL, Power BI  
**Purpose:** This notebook consolidates the work for **Week 1, Week 2 and Week 3** into one professional, reproducible workflow.

### What this notebook covers
- **Week 1:** Strategic planning, business objectives, KPI framework and initial logistics exploration.
- **Week 2:** Data collection, profiling, data-quality checks, cleaning, transformation and feature engineering.
- **Week 3:** Advanced EDA, logistics performance analysis, visualizations, analytical insights, root-cause analysis and recommendations.

> **Before running:** Keep `DataCoSupplyChainDataset.csv` in the same folder as this notebook, or change `DATA_PATH` in the loading cell.

# Week 1 — Strategic Planning and Data Exploration

## 1. Business Objective

The objective is to evaluate logistics and supply-chain performance using order, shipping, delivery, sales and profitability data. The analysis is designed to identify delivery delays, operational bottlenecks, shipping-mode differences, regional patterns and financial impact.

### Key business questions
1. What is the overall order, sales and profit performance?
2. What percentage of orders are at risk of late delivery?
3. Which shipping modes have the largest delivery-performance gaps?
4. Which markets, regions, categories and countries show higher late-delivery risk?
5. How does actual shipping time compare with scheduled shipping time?
6. What is the financial exposure associated with late orders?
7. Which operational areas should management prioritize for improvement?

## 2. Stakeholders
- Logistics / Operations managers
- Supply-chain planners
- Business analysts
- Finance / profitability teams
- Senior management

## 3. KPI Framework
- Total Orders
- Total Sales
- Total Profit
- Late Orders
- Late Delivery Rate
- Average Actual Shipping Days
- Average Scheduled Shipping Days
- Average Shipping Gap
- Estimated Late Orders

## 4. Analytical Methodology
**Raw data → Profiling → Cleaning → Transformation → KPI analysis → EDA → Visualization → Root-cause analysis → Recommendations**

This structure ensures that business conclusions are supported by validated data rather than isolated observations.

# 2. Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_PATH = "DataCoSupplyChainDataset.csv"

print("Libraries loaded successfully.")

# 3. Data Collection and Initial Inspection — Week 1/2

In [ ]:
df = pd.read_csv(DATA_PATH, encoding="latin1")

print("Dataset shape:", df.shape)
print("\nColumn count:", len(df.columns))
print("\nFirst 5 rows:")
display(df.head())

In [ ]:
print("Columns:")
display(pd.Series(df.columns, name="Column"))

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDataset information:")
df.info()

# Week 2 — Data Quality Assessment

The next section performs systematic checks for:
- missing values
- duplicate rows
- repeated Order IDs
- invalid data types
- date validity
- numerical consistency
- categorical consistency
- potential outliers

A repeated **Order ID is not automatically a duplicate row** because one order can legitimately have multiple records. The table grain must be considered before deleting anything.

In [ ]:
# Missing-value assessment
missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("missing_count")
)
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(2)

display(missing[missing["missing_count"] > 0])

In [ ]:
# Duplicate checks
print("Exact duplicate rows:", df.duplicated().sum())

if "Order Id" in df.columns:
    print("Repeated Order Id records:", df["Order Id"].duplicated().sum())
    print("Unique Order Ids:", df["Order Id"].nunique())

In [ ]:
# Descriptive statistics
numeric_summary = df.select_dtypes(include=np.number).describe().T
display(numeric_summary)

In [ ]:
# Categorical consistency
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

category_summary = []
for col in categorical_cols:
    category_summary.append({
        "column": col,
        "unique_values": df[col].nunique(dropna=True),
        "missing_values": df[col].isna().sum()
    })

display(pd.DataFrame(category_summary).sort_values("unique_values", ascending=False))

## 4. Data Cleaning and Standardization

The cleaning rules below are intentionally conservative:
- Preserve the raw dataset.
- Convert date fields to datetime.
- Strip accidental whitespace from text fields.
- Do not delete repeated Order IDs unless the full row is genuinely duplicated.
- Handle missing values according to business meaning instead of blindly replacing everything.
- Validate derived metrics after transformation.

In [ ]:
# Work on a copy so the raw dataframe remains available
clean_df = df.copy()

# Standardize text fields by removing leading/trailing whitespace
text_cols = clean_df.select_dtypes(include=["object"]).columns
for col in text_cols:
    clean_df[col] = clean_df[col].str.strip()

# Convert logistics date fields
date_cols = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]

for col in date_cols:
    if col in clean_df.columns:
        clean_df[col] = pd.to_datetime(clean_df[col], errors="coerce")

print("Cleaning and date conversion completed.")

In [ ]:
# Validate date conversion
date_check = pd.DataFrame({
    "column": date_cols,
    "missing_after_conversion": [
        clean_df[c].isna().sum() if c in clean_df.columns else np.nan
        for c in date_cols
    ]
})
display(date_check)

# 5. Feature Engineering

The following logistics features connect raw fields to business KPIs:
- `shipping_gap` = actual shipping days − scheduled shipping days
- `late_delivery_flag` = late-delivery indicator
- year/month/day fields for trend analysis
- order value and profit margin where the required fields are available

These fields will be reused throughout Week 3.

In [ ]:
# Shipping gap
if {"Days for shipping (real)", "Days for shipment (scheduled)"}.issubset(clean_df.columns):
    clean_df["shipping_gap"] = (
        clean_df["Days for shipping (real)"] -
        clean_df["Days for shipment (scheduled)"]
    )

# Time features
if "order date (DateOrders)" in clean_df.columns:
    clean_df["year"] = clean_df["order date (DateOrders)"].dt.year
    clean_df["month"] = clean_df["order date (DateOrders)"].dt.month
    clean_df["month_name"] = clean_df["order date (DateOrders)"].dt.month_name()
    clean_df["order_day"] = clean_df["order date (DateOrders)"].dt.day
    clean_df["order_date_only"] = clean_df["order date (DateOrders)"].dt.date

# Delivery flag
if "Late_delivery_risk" in clean_df.columns:
    clean_df["late_delivery_flag"] = clean_df["Late_delivery_risk"].astype(int)

# Optional order value / margin
if "Sales" in clean_df.columns and "Order Item Quantity" in clean_df.columns:
    clean_df["sales_per_item"] = (
        clean_df["Sales"] / clean_df["Order Item Quantity"].replace(0, np.nan)
    )

if {"Sales", "Order Profit Per Order"}.issubset(clean_df.columns):
    clean_df["profit_margin_pct"] = (
        clean_df["Order Profit Per Order"] /
        clean_df["Sales"].replace(0, np.nan) * 100
    )

display(clean_df.head())

In [ ]:
# Final preprocessing validation
print("Raw shape:", df.shape)
print("Processed shape:", clean_df.shape)
print("Exact duplicate rows after preprocessing:", clean_df.duplicated().sum())

print("\nMissing values in key logistics fields:")
key_cols = [
    "Order Id", "Sales", "Order Profit Per Order",
    "Days for shipping (real)", "Days for shipment (scheduled)",
    "Late_delivery_risk", "Delivery Status"
]
display(clean_df[[c for c in key_cols if c in clean_df.columns]].isna().sum().to_frame("missing"))

# Week 3 — Advanced Data Analysis & Visualization

Week 3 converts the cleaned dataset into deeper operational insights. Each visualization is followed by an interpretation section so the notebook explains **what the result means for the business**, not only what the chart looks like.

## 6. Core Logistics KPIs

In [ ]:
total_orders = clean_df["Order Id"].nunique() if "Order Id" in clean_df.columns else np.nan
total_sales = clean_df["Sales"].sum() if "Sales" in clean_df.columns else np.nan
total_profit = clean_df["Order Profit Per Order"].sum() if "Order Profit Per Order" in clean_df.columns else np.nan

late_orders = (
    clean_df.loc[clean_df["Late_delivery_risk"] == 1, "Order Id"].nunique()
    if {"Late_delivery_risk", "Order Id"}.issubset(clean_df.columns)
    else np.nan
)

late_rate = (
    clean_df["Late_delivery_flag"].mean() * 100
    if "late_delivery_flag" in clean_df.columns
    else np.nan
)

avg_actual = clean_df["Days for shipping (real)"].mean() if "Days for shipping (real)" in clean_df.columns else np.nan
avg_scheduled = clean_df["Days for shipment (scheduled)"].mean() if "Days for shipment (scheduled)" in clean_df.columns else np.nan
avg_gap = clean_df["shipping_gap"].mean() if "shipping_gap" in clean_df.columns else np.nan

kpis = pd.DataFrame({
    "KPI": [
        "Total Orders", "Total Sales", "Total Profit", "Late Orders",
        "Late Delivery Rate (%)", "Avg Actual Shipping Days",
        "Avg Scheduled Shipping Days", "Avg Shipping Gap"
    ],
    "Value": [
        total_orders, total_sales, total_profit, late_orders,
        late_rate, avg_actual, avg_scheduled, avg_gap
    ]
})
display(kpis)

## 7. Delivery Status Analysis

In [ ]:
delivery_status = (
    clean_df.groupby("Delivery Status")
    .agg(
        records=("Order Id", "count"),
        orders=("Order Id", "nunique"),
        sales=("Sales", "sum"),
        profit=("Order Profit Per Order", "sum")
    )
    .reset_index()
    .sort_values("orders", ascending=False)
)

display(delivery_status)

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(delivery_status["Delivery Status"], delivery_status["orders"])
plt.title("Orders by Delivery Status")
plt.xlabel("Delivery Status")
plt.ylabel("Unique Orders")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

display(delivery_status[["Delivery Status", "orders", "sales", "profit"]])

**Interpretation:** Use the table and chart above to compare the volume and financial contribution of each delivery-status group. A high volume of late/cancelled/at-risk orders indicates an operational area that should be investigated further rather than viewed only as a reporting metric.

## 8. Shipping Mode Performance

In [ ]:
shipping_analysis = (
    clean_df.groupby("Shipping Mode")
    .agg(
        orders=("Order Id", "nunique"),
        late_rate=("Late_delivery_risk", "mean"),
        avg_actual_days=("Days for shipping (real)", "mean"),
        avg_scheduled_days=("Days for shipment (scheduled)", "mean"),
        avg_gap=("shipping_gap", "mean"),
        sales=("Sales", "sum"),
        profit=("Order Profit Per Order", "sum")
    )
    .reset_index()
)

shipping_analysis["late_rate_pct"] = shipping_analysis["late_rate"] * 100
shipping_analysis = shipping_analysis.sort_values("late_rate_pct", ascending=False)
display(shipping_analysis)

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(shipping_analysis["Shipping Mode"], shipping_analysis["late_rate_pct"])
plt.title("Late Delivery Rate by Shipping Mode")
plt.xlabel("Shipping Mode")
plt.ylabel("Late Delivery Rate (%)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

**Interpretation:** The shipping mode with the highest late-delivery rate should be compared with its order volume, average shipping gap and financial contribution. This prevents management from focusing only on a high percentage from a very small number of orders.

## 9. Region and Market Analysis

In [ ]:
region_analysis = (
    clean_df.groupby("Order Region")
    .agg(
        orders=("Order Id", "nunique"),
        late_rate=("Late_delivery_risk", "mean"),
        avg_shipping_days=("Days for shipping (real)", "mean"),
        avg_gap=("shipping_gap", "mean"),
        sales=("Sales", "sum"),
        profit=("Order Profit Per Order", "sum")
    )
    .reset_index()
)
region_analysis["late_rate_pct"] = region_analysis["late_rate"] * 100

display(region_analysis.sort_values("late_rate_pct", ascending=False).head(15))

In [ ]:
market_analysis = (
    clean_df.groupby("Market")
    .agg(
        orders=("Order Id", "nunique"),
        late_rate=("Late_delivery_risk", "mean"),
        avg_shipping_days=("Days for shipping (real)", "mean"),
        avg_gap=("shipping_gap", "mean"),
        sales=("Sales", "sum"),
        profit=("Order Profit Per Order", "sum")
    )
    .reset_index()
)
market_analysis["late_rate_pct"] = market_analysis["late_rate"] * 100

display(market_analysis.sort_values("late_rate_pct", ascending=False))

In [ ]:
top_regions = region_analysis.sort_values("late_rate_pct", ascending=False).head(15)

plt.figure(figsize=(10, 6))
plt.barh(top_regions["Order Region"], top_regions["late_rate_pct"])
plt.title("Top Regions by Late Delivery Rate")
plt.xlabel("Late Delivery Rate (%)")
plt.ylabel("Order Region")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

**Interpretation:** Regional differences can indicate route complexity, capacity constraints, fulfillment-network issues or scheduling problems. Regions with both high order volume and high late-delivery rates deserve higher priority than low-volume regions with a high percentage alone.

## 10. Monthly Delivery Trend

In [ ]:
monthly_delivery = (
    clean_df.groupby(["year", "month"])
    .agg(
        orders=("Order Id", "nunique"),
        late_rate=("Late_delivery_risk", "mean"),
        avg_shipping_days=("Days for shipping (real)", "mean"),
        avg_gap=("shipping_gap", "mean"),
        sales=("Sales", "sum"),
        profit=("Order Profit Per Order", "sum")
    )
    .reset_index()
)

monthly_delivery["late_rate_pct"] = monthly_delivery["late_rate"] * 100
monthly_delivery["period"] = (
    monthly_delivery["year"].astype(str) + "-" +
    monthly_delivery["month"].astype(str).str.zfill(2)
)

display(monthly_delivery.sort_values("period"))

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly_delivery["period"], monthly_delivery["late_rate_pct"], marker="o")
plt.title("Monthly Late Delivery Rate Trend")
plt.xlabel("Period")
plt.ylabel("Late Delivery Rate (%)")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

**Interpretation:** Monthly changes help identify periods where delivery performance deteriorates or improves. Peaks should be investigated against order volume, shipping mode, region and category to identify the underlying operational driver.

## 11. Shipping Gap Analysis

In [ ]:
gap_by_mode = (
    clean_df.groupby("Shipping Mode")["shipping_gap"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
    .sort_values("mean", ascending=False)
)

display(gap_by_mode)

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(clean_df["shipping_gap"].dropna(), bins=25)
plt.title("Distribution of Shipping Gap")
plt.xlabel("Actual Shipping Days − Scheduled Shipping Days")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()

**Interpretation:** A positive shipping gap means actual shipping duration exceeded the scheduled duration. The distribution helps distinguish isolated delays from a broader operational pattern.

## 12. Category and Customer-Segment Analysis

In [ ]:
category_analysis = (
    clean_df.groupby("Category Name")
    .agg(
        orders=("Order Id", "nunique"),
        late_rate=("Late_delivery_risk", "mean"),
        sales=("Sales", "sum"),
        avg_profit=("Order Profit Per Order", "mean"),
        avg_gap=("shipping_gap", "mean")
    )
    .reset_index()
)
category_analysis["late_rate_pct"] = category_analysis["late_rate"] * 100

display(category_analysis.sort_values("late_rate_pct", ascending=False).head(15))

In [ ]:
customer_segment = (
    clean_df.groupby("Customer Segment")
    .agg(
        orders=("Order Id", "nunique"),
        late_rate=("Late_delivery_risk", "mean"),
        sales=("Sales", "sum"),
        profit=("Order Profit Per Order", "sum")
    )
    .reset_index()
)
customer_segment["late_rate_pct"] = customer_segment["late_rate"] * 100

display(customer_segment)

In [ ]:
top_categories = category_analysis.sort_values("late_rate_pct", ascending=False).head(12)

plt.figure(figsize=(10, 6))
plt.barh(top_categories["Category Name"], top_categories["late_rate_pct"])
plt.title("Categories with Higher Late Delivery Rates")
plt.xlabel("Late Delivery Rate (%)")
plt.ylabel("Category")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

**Interpretation:** Category-level differences can reveal products that require different fulfillment handling, have higher shipment complexity, or experience recurring operational delays. Customer-segment results help determine whether service issues are concentrated in a particular customer group.

## 13. Shipping Mode × Region Root-Cause Analysis

In [ ]:
mode_region = (
    clean_df.groupby(["Shipping Mode", "Order Region"])
    .agg(
        orders=("Order Id", "nunique"),
        late_rate=("Late_delivery_risk", "mean"),
        avg_gap=("shipping_gap", "mean"),
        avg_actual_days=("Days for shipping (real)", "mean"),
        avg_scheduled_days=("Days for shipment (scheduled)", "mean"),
        sales=("Sales", "sum"),
        profit=("Order Profit Per Order", "sum")
    )
    .reset_index()
)
mode_region["late_rate_pct"] = mode_region["late_rate"] * 100

# Prioritize combinations with meaningful volume
root_cause_candidates = mode_region.sort_values(
    ["late_rate_pct", "orders"], ascending=[False, False]
).head(20)

display(root_cause_candidates)

**Interpretation:** The shipping-mode/region combination is a useful root-cause lens because a poor overall result may be driven by a smaller number of specific operational combinations. Prioritize combinations that show both high late-delivery risk and meaningful order volume.

## 14. Financial Impact of Late Orders

In [ ]:
late_orders_df = clean_df[clean_df["Late_delivery_risk"] == 1].copy()

late_sales = late_orders_df["Sales"].sum()
total_sales = clean_df["Sales"].sum()
late_sales_share = (late_sales / total_sales * 100) if total_sales else np.nan

late_profit = late_orders_df["Order Profit Per Order"].sum()
total_profit_value = clean_df["Order Profit Per Order"].sum()

financial_impact = pd.DataFrame({
    "Metric": [
        "Late-order sales",
        "Total sales",
        "Late-order sales share (%)",
        "Late-order profit",
        "Total profit"
    ],
    "Value": [
        late_sales,
        total_sales,
        late_sales_share,
        late_profit,
        total_profit_value
    ]
})

display(financial_impact)

**Interpretation:** Late delivery is not only an operational KPI. The sales and profit associated with late orders indicate the financial importance of fixing delivery-performance issues.

## 15. Correlation Analysis

Correlation is used as an exploratory tool, not as proof of causation. It can highlight numeric variables that move together and deserve further investigation.

In [ ]:
corr_cols = [
    c for c in [
        "Sales", "Order Profit Per Order",
        "Days for shipping (real)", "Days for shipment (scheduled)",
        "shipping_gap", "Late_delivery_risk",
        "Order Item Quantity", "Order Item Discount Rate"
    ] if c in clean_df.columns
]

corr_matrix = clean_df[corr_cols].corr(numeric_only=True)
display(corr_matrix.round(2))

In [ ]:
plt.figure(figsize=(9, 7))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=70, ha="right")
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
plt.title("Correlation Matrix — Logistics Variables")
plt.tight_layout()
plt.show()

**Interpretation:** Stronger positive or negative relationships can be used to select variables for deeper investigation. Correlation alone does not establish that one logistics factor causes another.

# 16. Automated Key Findings

This section converts the analysis tables into concise, data-driven observations. The exact values will update automatically when the notebook is run with the dataset.

In [ ]:
# Dynamic findings
if not shipping_analysis.empty:
    worst_mode = shipping_analysis.iloc[0]
    print(
        f"1. Highest late-delivery-rate shipping mode: {worst_mode['Shipping Mode']} "
        f"({worst_mode['late_rate_pct']:.2f}%)."
    )

if not region_analysis.empty:
    worst_region = region_analysis.sort_values("late_rate_pct", ascending=False).iloc[0]
    print(
        f"2. Highest late-delivery-rate region: {worst_region['Order Region']} "
        f"({worst_region['late_rate_pct']:.2f}%)."
    )

if not monthly_delivery.empty:
    peak_month = monthly_delivery.sort_values("late_rate_pct", ascending=False).iloc[0]
    print(
        f"3. Highest monthly late-delivery rate: {peak_month['period']} "
        f"({peak_month['late_rate_pct']:.2f}%)."
    )

if not category_analysis.empty:
    worst_category = category_analysis.sort_values("late_rate_pct", ascending=False).iloc[0]
    print(
        f"4. Highest category late-delivery rate: {worst_category['Category Name']} "
        f"({worst_category['late_rate_pct']:.2f}%)."
    )

print(f"5. Overall late-delivery rate: {late_rate:.2f}%")
print(f"6. Average shipping gap: {avg_gap:.2f} days")
print(f"7. Late-order sales share: {late_sales_share:.2f}%")

# 17. Business Recommendations

Based on the analytical framework, recommendations should be tied directly to the highest-impact findings:

1. **Prioritize high-risk shipping modes:** Review capacity, carrier allocation, dispatch planning and service-level targets for modes with consistently high late-delivery rates.
2. **Target high-volume/high-risk regions:** Regional interventions should focus first on areas where both order volume and delay risk are significant.
3. **Monitor shipping gap:** Use actual-vs-scheduled shipping performance as an operational control metric.
4. **Investigate recurring monthly peaks:** Compare high-delay periods with demand volume, shipping mode and regional mix.
5. **Review category-specific fulfillment:** Categories with repeated delays may need different handling, packaging or fulfillment planning.
6. **Track financial exposure:** Monitor sales and profit linked to late orders so operational improvements can be prioritized by business impact.
7. **Maintain a repeatable data pipeline:** Keep raw data, cleaned data, SQL analysis and Power BI reporting logically separated for reproducibility.

# 18. Power BI Handoff

The processed dataset and KPI definitions are ready to support the logistics dashboard.

### Recommended dashboard pages
**Page 1 — Executive Overview**
- Total Orders
- Total Sales
- Total Profit
- Late Orders
- Late Delivery Rate

**Page 2 — Delivery Performance**
- Actual vs Scheduled Shipping Days
- Shipping Gap
- Late Delivery by Shipping Mode
- Delivery Status

**Page 3 — Geographic Performance**
- Market
- Region
- Country
- Late Delivery Rate

**Page 4 — Root Cause & Profitability**
- Shipping Mode × Region
- Category performance
- Late-order sales/profit impact

The definitions used in Python should remain consistent with Power BI measures.

# 19. Export Analysis-Ready Dataset

Run this cell after validating the final data. The exported CSV can be used for PostgreSQL or Power BI.

In [ ]:
OUTPUT_PATH = "Logistics_Cleaned_Week1_Week3.csv"

clean_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved cleaned dataset to: {OUTPUT_PATH}")

# 20. Week 1–3 Completion Checklist

### Week 1
- [x] Strategic planning
- [x] Business objectives
- [x] Stakeholder identification
- [x] KPI framework
- [x] Logistics analytical questions
- [x] Initial data exploration

### Week 2
- [x] Data collection/loading
- [x] Data profiling
- [x] Missing-value checks
- [x] Duplicate checks
- [x] Data-type validation
- [x] Date conversion
- [x] Data cleaning
- [x] Feature engineering
- [x] Final validation

### Week 3
- [x] Advanced EDA
- [x] Shipping-mode analysis
- [x] Region/market analysis
- [x] Delivery-status analysis
- [x] Monthly trend analysis
- [x] Shipping-gap analysis
- [x] Category/customer-segment analysis
- [x] Root-cause analysis
- [x] Financial impact analysis
- [x] Visualizations
- [x] Visualization interpretation
- [x] Business recommendations

> **Important:** Run all cells with the actual DataCo CSV before submitting. The notebook is designed to calculate the final numerical results from the dataset rather than using invented values.